# Imports

In [1]:
import numpy as np
from pathlib import Path
import pandas as pd
import pm4py
import pysubgroup as ps_orig
from scipy import stats

# Loading Data

In [2]:
domestic = pm4py.read_xes("Dataset BPIC 2020/DomesticDeclarations.xes")
international = pm4py.read_xes("Dataset BPIC 2020/InternationalDeclarations.xes")
permits = pm4py.read_xes("Dataset BPIC 2020/PermitLog.xes")

C:\Users\20200604\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pm4py\utils.py:795: UserWarning: In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.
  warnings.warn("In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.")


In [3]:
bpic2019 = pm4py.read_xes("Dataset BPIC 2019/BPI_Challenge_2019.xes")

# Data preparation

## BPIC 2020 Declarations log

In [4]:
domestic["provenance"] = "domestic"
international["provenance"] = "international"
declarations = pd.concat([domestic, international], ignore_index=True)
start = declarations[declarations["concept:name"] == "Declaration SUBMITTED by EMPLOYEE"].groupby("case:concept:name")["time:timestamp"].min()
end = declarations[declarations["concept:name"] == "Payment Handled"].groupby("case:concept:name")["time:timestamp"].max()
throughput = (end - start)
declarations["throughput"] = declarations["case:concept:name"].map(throughput)


In [5]:
activities = ['Declaration SUBMITTED by EMPLOYEE', 'Declaration FINAL_APPROVED by SUPERVISOR', 'Request Payment', 'Payment Handled', 'Declaration APPROVED by PRE_APPROVER', 'Declaration REJECTED by MISSING', 'Declaration REJECTED by PRE_APPROVER', 'Declaration REJECTED by EMPLOYEE', 'Declaration SAVED by EMPLOYEE', 'Declaration REJECTED by SUPERVISOR', 'Declaration APPROVED by ADMINISTRATION', 'Declaration APPROVED by BUDGET OWNER', 'Declaration FOR_APPROVAL by SUPERVISOR', 'Declaration REJECTED by ADMINISTRATION', 'Declaration FOR_APPROVAL by PRE_APPROVER', 'Declaration REJECTED by BUDGET OWNER', 'Declaration FOR_APPROVAL by ADMINISTRATION', 'Start trip', 'End trip', 'Permit SUBMITTED by EMPLOYEE', 'Permit FINAL_APPROVED by SUPERVISOR', 'Permit APPROVED by SUPERVISOR', 'Permit FINAL_APPROVED by DIRECTOR', 'Permit APPROVED by PRE_APPROVER', 'Permit REJECTED by MISSING', 'Declaration APPROVED by SUPERVISOR', 'Declaration FINAL_APPROVED by DIRECTOR', 'Permit REJECTED by PRE_APPROVER', 'Permit REJECTED by EMPLOYEE', 'Declaration REJECTED by DIRECTOR', 'Permit REJECTED by SUPERVISOR', 'Permit APPROVED by ADMINISTRATION', 'Send Reminder', 'Permit APPROVED by BUDGET OWNER', 'Permit REJECTED by ADMINISTRATION', 'Permit REJECTED by BUDGET OWNER', 'Permit REJECTED by DIRECTOR']

declarations_feat = declarations.groupby("case:id").agg({
    "provenance": "first",
    "case:Amount": "first",
    "case:BudgetNumber": "first",
    "case:Permit ProjectNumber": "first",
    "case:Permit OrganizationalEntity": "first",
    "throughput": "first",
}).reset_index()

activity_flags = declarations.groupby("case:id")["concept:name"].apply(lambda x: set(x))
for act in activities:
    declarations_feat[act] = declarations_feat["case:id"].map(activity_flags).apply(lambda s: int(act in s))


In [6]:
declarations_feat = declarations_feat[declarations_feat["throughput"].notna()]
declarations_feat["throughput_num"] = declarations_feat["throughput"].dt.total_seconds() / 86400
declarations_feat[["case:Permit ProjectNumber", "case:Permit OrganizationalEntity"]] = declarations_feat[["case:Permit ProjectNumber", "case:Permit OrganizationalEntity"]].fillna("not available")

## BPIC 2020 permits log

In [7]:
permits_clean = permits[["case:concept:name", "case:BudgetNumber", "case:ProjectNumber", "case:OrganizationalEntity", "case:Overspent", "case:RequestedBudget", "case:OverspentAmount"]]
permits_clean = permits_clean.drop_duplicates()
permits_clean = permits_clean[permits_clean["case:RequestedBudget"]!= 0]

## BPIC 2019 3-way matching log

In [8]:
bpic2019 = bpic2019[["case:concept:name"] + [c for c in bpic2019.columns if c != "case:concept:name"]]
invoicing_3way = bpic2019[(bpic2019["case:Item Category"] == "3-way match, invoice before GR") | (bpic2019["case:Item Category"] == "3-way match, invoice after GR")]


In [9]:
invoicing_3way = invoicing_3way.sort_values("time:timestamp")

def resource_type(r):
    if r == "NONE":
        return "NONE"
    elif str(r).startswith("user_"):
        return "user"
    elif str(r).startswith("batch_"):
        return "batch"

invoicing_3way["_res_type"] = invoicing_3way["org:resource"].apply(resource_type)

invoicing_3way_grouped = invoicing_3way.groupby("case:concept:name")

res_perc = invoicing_3way_grouped["_res_type"].value_counts(normalize=True).unstack(fill_value=0) * 100
res_perc = res_perc.reindex(columns=["NONE", "user", "batch"], fill_value=0)
res_perc.columns = ["resource_NONE_perc", "resource_user_perc", "resource_batch_perc"]

throughput = invoicing_3way_grouped["time:timestamp"].agg(lambda x: (x.max() - x.min()).total_seconds() / 86400)
throughput.name = "throughput"

path = invoicing_3way_grouped["concept:name"].agg(list)
path.name = "path"

first_cols = [
    "case:Item Category", "case:Sub spend area text", "case:Spend area text",
    "case:Purch. Doc. Category name", "case:Vendor", "case:Company", "case:Item",
    "Cumulative net worth (EUR)", "case:Item Type", "case:Document Type", "case:Source",
    "case:Purchasing Document", "case:Spend classification text"
]
firsts = invoicing_3way_grouped[first_cols].first()

invoicing_3way_feat = firsts.join(res_perc).join(throughput).join(path).reset_index()

In [10]:
hp = ['Create Purchase Order Item', 'Vendor creates invoice', 'Record Goods Receipt', 'Record Invoice Receipt', 'Clear Invoice']
hp_set = set(hp)

def is_subsequence(sub, path):
    it = iter(path)
    return all(a in it for a in sub)

def collapse_redos(path):
    out = []
    for a in path:
        if not out or out[-1] != a:
            out.append(a)
    return out


invoicing_3way_feat["has_hp_act_inc"] = invoicing_3way_feat["path"].apply(lambda p: hp_set.issubset(set(p)))
invoicing_3way_feat["has_hp_act_exc"] = invoicing_3way_feat["path"].apply(lambda p: set(p) == hp_set)
invoicing_3way_feat["hp_order_inc"] = invoicing_3way_feat["path"].apply(lambda p: is_subsequence(hp, p))
invoicing_3way_feat["hp_order_redo_exc"] = invoicing_3way_feat["path"].apply(lambda p: collapse_redos(p) == hp)
invoicing_3way_feat["hp_order_exact"] = invoicing_3way_feat["path"].apply(lambda p: p == hp)

invoicing_3way_feat["CPOI_num"] = invoicing_3way_feat["path"].apply(lambda p: p.count('Create Purchase Order Item'))
invoicing_3way_feat["VCI_num"] = invoicing_3way_feat["path"].apply(lambda p: p.count('Vendor creates invoice'))
invoicing_3way_feat["RGR_num"] = invoicing_3way_feat["path"].apply(lambda p: p.count('Record Goods Receipt'))
invoicing_3way_feat["RIR_num"] = invoicing_3way_feat["path"].apply(lambda p: p.count('Record Invoice Receipt'))
invoicing_3way_feat["CI_num"] = invoicing_3way_feat["path"].apply(lambda p: p.count('Clear Invoice'))


In [11]:
majority = invoicing_3way_feat[["resource_NONE_perc", "resource_user_perc", "resource_batch_perc"]].idxmax(axis=1).map({
    "resource_NONE_perc": "NONE",
    "resource_user_perc": "user",
    "resource_batch_perc": "batch"
})
invoicing_3way_feat.insert(invoicing_3way_feat.columns.get_loc("resource_batch_perc") + 1, "resource_majority", majority)

In [12]:
invoicing_3way_feat_filtered = invoicing_3way_feat[(invoicing_3way_feat["CI_num"]>0) & ((invoicing_3way_feat["RGR_num"]>0) | (invoicing_3way_feat["RIR_num"]>0))]

In [13]:
freq_cols = ["case:Vendor", "case:Sub spend area text", "case:Spend area text", "case:Item"]

for col in freq_cols:
    vc = invoicing_3way_feat_filtered[col].value_counts()
    infrequent = vc[vc <= 50].index
    invoicing_3way_feat_filtered[col] = invoicing_3way_feat_filtered[col].where(~invoicing_3way_feat_filtered[col].isin(infrequent), "infrequent")

small_max, medium_max = 3, 14
vc = invoicing_3way_feat_filtered["case:Purchasing Document"].value_counts()
order_size_map = vc.apply(lambda n: "small" if n <= small_max else ("medium" if n <= medium_max else "large"))
invoicing_3way_feat_filtered["order_size"] = invoicing_3way_feat_filtered["case:Purchasing Document"].map(order_size_map)

In [14]:
binary_cols = ["has_hp_act_exc", "hp_order_inc", "hp_order_redo_exc", "hp_order_exact"]
joint = invoicing_3way_feat_filtered.groupby(binary_cols).size().reset_index(name="count")
joint["percent"] = joint["count"] / joint["count"].sum() * 100
joint = joint.sort_values("count", ascending=False).reset_index(drop=True)
joint

,has_hp_act_exc,hp_order_inc,hp_order_redo_exc,hp_order_exact,count,percent
0,False,False,False,False,50748,27.731451
1,True,True,True,True,50283,27.477349
2,False,True,False,False,46873,25.613941
3,True,False,False,False,33364,18.231893
4,True,True,False,False,886,0.484158
5,True,True,True,False,844,0.461207


In [15]:
def conformance_severity(row):
    if row["hp_order_exact"]:
        return 0
    if row["hp_order_redo_exc"]:
        return 1
    if row["has_hp_act_exc"] and row["hp_order_inc"]:
        return 2
    if row["hp_order_inc"]:
        return 3
    if row["has_hp_act_exc"]:
        return 4
    return 5

invoicing_3way_feat_filtered["conformance_severity"] = invoicing_3way_feat_filtered.apply(conformance_severity, axis=1)
print(invoicing_3way_feat_filtered["conformance_severity"].value_counts().sort_index())

conformance_severity
0    50283
1      844
2      886
3    46873
4    33364
5    50748
Name: count, dtype: int64


# Helpers

In [16]:
def dedup_by_cover(result, emm_result):
    """Drop subgroups whose covered row-set duplicates a better-scoring one."""
    descriptions = [sg for _, sg in result.to_descriptions()]
    seen = set()
    keep_mask = []
    for sg in descriptions:
        cover_arr, _ = ps_orig.get_cover_array_and_size(sg, len(data), data)
        sig = hash(np.packbits(np.asarray(cover_arr)).tobytes())
        keep_mask.append(sig not in seen)
        seen.add(sig)

    emm_result = emm_result[keep_mask].reset_index(drop=True)

    return emm_result[emm_result["quality"]>0]

# BPIC 2020 Experiments

## Throughput-Time Search

### Experiment 1

In [17]:
data = declarations_feat
feat_to_keep = ["provenance","case:BudgetNumber", "case:Permit ProjectNumber", "case:Permit OrganizationalEntity", "case:Amount"]
ignore = [c for c in data.columns if c not in feat_to_keep]
target = ps_orig.NumericTarget("throughput_num")
searchspace = ps_orig.create_selectors(data, ignore=ignore)
task = ps_orig.SubgroupDiscoveryTask (
    data,
    target,
    searchspace,
    result_set_size=10,
    depth=1,
    qf=ps_orig.StandardQFNumeric(a=0.5,min_sg_coverage=0.02, max_sg_coverage=0.8))
result_exp_1 = ps_orig.BeamSearch(beam_width_adaptive = True).execute(task)
emm_result_exp_1 = result_exp_1.to_dataframe()
emm_result_exp_1_dedup = dedup_by_cover(result_exp_1, emm_result_exp_1)

In [28]:
emm_result_exp_1

,quality,subgroup,size_sg,size_dataset,mean_sg,mean_dataset,std_sg,std_dataset,median_sg,median_dataset,max_sg,max_dataset,min_sg,min_dataset,mean_lift,median_lift
0,1.570722,case:Amount>=545.30,3246,16230,16.269843,12.757601,17.576034,16.447325,11.242164,8.398119,272.201134,429.112824,1.137905,1.064931,1.275306,1.338653
1,1.258028,case:Permit OrganizationalEntity=='organizatio...,840,16230,18.287408,12.757601,18.117797,16.447325,14.158252,8.398119,349.814306,429.112824,1.985602,1.064931,1.433452,1.685884
2,1.142618,provenance=='international',6187,16230,14.608233,12.757601,17.355072,16.447325,10.245301,8.398119,429.112824,429.112824,1.087350,1.064931,1.145061,1.219952
3,0.925295,case:Amount: [28.31:55.67[,3246,16230,10.688579,12.757601,16.017803,16.447325,7.286991,8.398119,429.112824,429.112824,1.064931,1.064931,0.837820,0.867693
4,0.896828,provenance=='domestic',10043,16230,11.617517,12.757601,15.754425,16.447325,7.411308,8.398119,368.193449,429.112824,1.064931,1.064931,0.910635,0.882496
5,0.896828,case:Permit ProjectNumber=='not available',10043,16230,11.617517,12.757601,15.754425,16.447325,7.411308,8.398119,368.193449,429.112824,1.064931,1.064931,0.910635,0.882496
6,0.896828,case:Permit OrganizationalEntity=='not available',10043,16230,11.617517,12.757601,15.754425,16.447325,7.411308,8.398119,368.193449,429.112824,1.064931,1.064931,0.910635,0.882496
7,0.896828,case:BudgetNumber=='budget 86566',10043,16230,11.617517,12.757601,15.754425,16.447325,7.411308,8.398119,368.193449,429.112824,1.064931,1.064931,0.910635,0.882496
8,0.635416,case:Permit ProjectNumber=='UNKNOWN',2231,16230,14.471431,12.757601,18.695199,16.447325,9.997975,8.398119,349.814306,429.112824,1.194502,1.064931,1.134338,1.190502
9,0.594998,case:Amount<28.31,3246,16230,11.427145,12.757601,18.321454,16.447325,7.296360,8.398119,349.814306,429.112824,1.258727,1.064931,0.895713,0.868809


In [33]:
print(stats.percentileofscore(declarations_feat["case:Amount"], 545.30))

80.0


### Experiment 2a

In [19]:
data = declarations_feat
ignore = ["throughput","throughput_num"]
target = ps_orig.NumericTarget("throughput_num")
searchspace = ps_orig.create_selectors(data, ignore=ignore)
task = ps_orig.SubgroupDiscoveryTask (
    data,
    target,
    searchspace,
    result_set_size=10,
    depth=2,
    qf=ps_orig.StandardQFNumeric(a=0.5,min_sg_coverage=0.02, max_sg_coverage=0.8))
result_exp_2a = ps_orig.BeamSearch(beam_width_adaptive = True).execute(task)
emm_result_exp_2a = result_exp_2a.to_dataframe()
emm_result_exp_2a_dedup = dedup_by_cover(result_exp_2a, emm_result_exp_2a)

In [36]:
emm_result_exp_2a

,quality,subgroup,size_sg,size_dataset,mean_sg,mean_dataset,std_sg,std_dataset,median_sg,median_dataset,max_sg,max_dataset,min_sg,min_dataset,mean_lift,median_lift
0,3.373171,Declaration APPROVED by ADMINISTRATION==1 AND ...,2040,16230,22.272024,12.757601,25.084458,16.447325,14.904161,8.398119,368.193449,429.112824,1.769549,1.064931,1.745785,1.774702
1,3.372456,Declaration REJECTED by EMPLOYEE==1,2240,16230,21.835422,12.757601,24.631767,16.447325,14.272564,8.398119,368.193449,429.112824,1.769549,1.064931,1.711562,1.699495
2,3.372456,Declaration REJECTED by EMPLOYEE==1 AND Permit...,2240,16230,21.835422,12.757601,24.631767,16.447325,14.272564,8.398119,368.193449,429.112824,1.769549,1.064931,1.711562,1.699495
3,3.372456,Declaration REJECTED by EMPLOYEE==1 AND Permit...,2240,16230,21.835422,12.757601,24.631767,16.447325,14.272564,8.398119,368.193449,429.112824,1.769549,1.064931,1.711562,1.699495
4,3.372456,Declaration REJECTED by EMPLOYEE==1 AND Paymen...,2240,16230,21.835422,12.757601,24.631767,16.447325,14.272564,8.398119,368.193449,429.112824,1.769549,1.064931,1.711562,1.699495
5,3.372456,Declaration REJECTED by EMPLOYEE==1 AND Declar...,2240,16230,21.835422,12.757601,24.631767,16.447325,14.272564,8.398119,368.193449,429.112824,1.769549,1.064931,1.711562,1.699495
6,3.372456,Declaration REJECTED by EMPLOYEE==1 AND Declar...,2240,16230,21.835422,12.757601,24.631767,16.447325,14.272564,8.398119,368.193449,429.112824,1.769549,1.064931,1.711562,1.699495
7,3.372456,Declaration FOR_APPROVAL by SUPERVISOR==0 AND ...,2240,16230,21.835422,12.757601,24.631767,16.447325,14.272564,8.398119,368.193449,429.112824,1.769549,1.064931,1.711562,1.699495
8,3.372456,Declaration FOR_APPROVAL by PRE_APPROVER==0 AN...,2240,16230,21.835422,12.757601,24.631767,16.447325,14.272564,8.398119,368.193449,429.112824,1.769549,1.064931,1.711562,1.699495
9,3.372299,Declaration REJECTED by EMPLOYEE==1 AND Permit...,2214,16230,21.888142,12.757601,24.726978,16.447325,14.269236,8.398119,368.193449,429.112824,1.769549,1.064931,1.715694,1.699099


In [20]:
emm_result_exp_2a_dedup

,quality,subgroup,size_sg,size_dataset,mean_sg,mean_dataset,std_sg,std_dataset,median_sg,median_dataset,max_sg,max_dataset,min_sg,min_dataset,mean_lift,median_lift
0,3.373171,Declaration APPROVED by ADMINISTRATION==1 AND ...,2040,16230,22.272024,12.757601,25.084458,16.447325,14.904161,8.398119,368.193449,429.112824,1.769549,1.064931,1.745785,1.774702
1,3.372456,Declaration REJECTED by EMPLOYEE==1,2240,16230,21.835422,12.757601,24.631767,16.447325,14.272564,8.398119,368.193449,429.112824,1.769549,1.064931,1.711562,1.699495
2,3.372299,Declaration REJECTED by EMPLOYEE==1 AND Permit...,2214,16230,21.888142,12.757601,24.726978,16.447325,14.269236,8.398119,368.193449,429.112824,1.769549,1.064931,1.715694,1.699099


### Experiment 2b

In [21]:
data = declarations_feat[~((declarations_feat["Declaration REJECTED by EMPLOYEE"] == 1) & (declarations_feat["Declaration APPROVED by ADMINISTRATION"] == 1))]
feat_to_keep = ["provenance","case:BudgetNumber", "case:Permit ProjectNumber", "case:Permit OrganizationalEntity","case:Amount"]
ignore = [c for c in data.columns if c not in feat_to_keep]
target = ps_orig.NumericTarget("throughput_num")
searchspace = ps_orig.create_selectors(data, ignore=ignore)
task = ps_orig.SubgroupDiscoveryTask (
    data,
    target,
    searchspace,
    result_set_size=10,
    depth=1,
    qf=ps_orig.StandardQFNumeric(a=0.5,min_sg_coverage=0.02, max_sg_coverage=0.8))
result_exp_2b = ps_orig.BeamSearch(beam_width_adaptive = True).execute(task)
emm_result_exp_2b = result_exp_2b.to_dataframe()
emm_result_exp_2b_dedup = dedup_by_cover(result_exp_2b, emm_result_exp_2b)

In [29]:
emm_result_exp_2b

,quality,subgroup,size_sg,size_dataset,mean_sg,mean_dataset,std_sg,std_dataset,median_sg,median_dataset,max_sg,max_dataset,min_sg,min_dataset,mean_lift,median_lift
0,0.943422,case:Amount>=447.36,2838,14190,13.499332,11.389777,14.463677,14.284936,10.093501,7.963241,272.201134,429.112824,1.137905,1.064931,1.185215,1.267512
1,0.837533,provenance=='international',4968,14190,12.805252,11.389777,16.199833,14.284936,9.273119,7.963241,429.112824,429.112824,1.087350,1.064931,1.124276,1.164491
2,0.730258,case:Permit OrganizationalEntity=='organizatio...,596,14190,14.953012,11.389777,16.772492,14.284936,12.768918,7.963241,349.814306,429.112824,1.985602,1.064931,1.312845,1.603483
3,0.614724,provenance=='domestic',9222,14190,10.627244,11.389777,13.074830,14.284936,7.292922,7.963241,344.318148,429.112824,1.064931,1.064931,0.933051,0.915823
4,0.614724,case:Permit ProjectNumber=='not available',9222,14190,10.627244,11.389777,13.074830,14.284936,7.292922,7.963241,344.318148,429.112824,1.064931,1.064931,0.933051,0.915823
5,0.614724,case:Permit OrganizationalEntity=='not available',9222,14190,10.627244,11.389777,13.074830,14.284936,7.292922,7.963241,344.318148,429.112824,1.064931,1.064931,0.933051,0.915823
6,0.614724,case:BudgetNumber=='budget 86566',9222,14190,10.627244,11.389777,13.074830,14.284936,7.292922,7.963241,344.318148,429.112824,1.064931,1.064931,0.933051,0.915823
7,0.564261,case:Amount: [26.77:50.40[,2838,14190,10.128052,11.389777,14.989601,14.284936,7.274884,7.963241,429.112824,429.112824,1.064931,1.064931,0.889223,0.913558
8,0.441855,case:Amount: [50.40:126.89[,2838,14190,10.401759,11.389777,10.850828,14.284936,7.272870,7.963241,182.249063,429.112824,1.120313,1.064931,0.913254,0.913305
9,0.356477,case:Permit ProjectNumber=='UNKNOWN',1737,14190,12.408656,11.389777,17.400771,14.284936,8.251794,7.963241,349.814306,429.112824,1.194502,1.064931,1.089456,1.036236


In [34]:
print(stats.percentileofscore(declarations_feat[~((declarations_feat["Declaration REJECTED by EMPLOYEE"] == 1) & (declarations_feat["Declaration APPROVED by ADMINISTRATION"] == 1))]["case:Amount"], 447.36))

80.0


## Overspending Search (Experiment 3)

In [23]:
data = permits_clean
ignore = ["case:concept:name", "case:Overspent", "case:OverspentAmount"]
target = ps_orig.BinaryTarget("case:Overspent", target_value =True)
searchspace = ps_orig.create_selectors(data, ignore=ignore)
task = ps_orig.SubgroupDiscoveryTask (
    data,
    target,
    searchspace,
    result_set_size=10,
    depth=1,
    qf=ps_orig.StandardQF(a=0.5,min_sg_coverage=0.02, max_sg_coverage=0.8))
result_exp_3 = ps_orig.BeamSearch().execute(task)
emm_result_exp_3 = result_exp_3.to_dataframe()
emm_result_exp_3_dedup = dedup_by_cover(result_exp_3, emm_result_exp_3)

In [24]:
emm_result_exp_3_dedup

,quality,subgroup,size_sg,size_dataset,positives_sg,positives_dataset,size_complement,relative_size_sg,relative_size_complement,coverage_sg,coverage_complement,target_share_sg,target_share_complement,target_share_dataset,lift
0,0.023452,case:OrganizationalEntity=='organizational uni...,564,6763,201,1861,6199,0.083395,0.916605,0.108006,0.891994,0.356383,0.267785,0.275174,1.295120
1,0.023440,case:OrganizationalEntity=='organizational uni...,523,6763,188,1861,6240,0.077333,0.922667,0.101021,0.898979,0.359465,0.268109,0.275174,1.306319
2,0.014134,case:OrganizationalEntity=='organizational uni...,973,6763,304,1861,5790,0.143871,0.856129,0.163353,0.836647,0.312436,0.268912,0.275174,1.135413
3,0.012225,case:RequestedBudget: [664.90:1121.34[,1352,6763,409,1861,5411,0.199911,0.800089,0.219774,0.780226,0.302515,0.268342,0.275174,1.099359
4,0.006724,case:OrganizationalEntity=='organizational uni...,255,6763,79,1861,6508,0.037705,0.962295,0.042450,0.957550,0.309804,0.273817,0.275174,1.125848
5,0.006098,case:OrganizationalEntity=='organizational uni...,358,6763,108,1861,6405,0.052935,0.947065,0.058033,0.941967,0.301676,0.273692,0.275174,1.096311
6,0.005947,case:OrganizationalEntity=='organizational uni...,338,6763,102,1861,6425,0.049978,0.950022,0.054809,0.945191,0.301775,0.273774,0.275174,1.096671
7,0.005848,case:RequestedBudget: [1121.34:1941.24[,1353,6763,390,1861,5410,0.200059,0.799941,0.209565,0.790435,0.288248,0.271904,0.275174,1.047514
8,0.005099,case:BudgetNumber=='budget 698',252,6763,76,1861,6511,0.037262,0.962738,0.040838,0.959162,0.301587,0.274151,0.275174,1.095989
9,0.003776,case:BudgetNumber=='budget 6198',209,6763,62,1861,6554,0.030903,0.969097,0.033315,0.966685,0.296651,0.274489,0.275174,1.078049


# BPIC 2019 Experiment 4: Conformance-Deviation Search

In [17]:
data = invoicing_3way_feat_filtered
ignore = ["case:concept:name","case:Purchasing Document","path","has_hp_act_inc", "has_hp_act_exc","hp_order_inc","hp_order_redo_exc","hp_order_exact", "CPOI_num","VCI_num","RGR_num","RIR_num","CI_num", "act_to_first_dev_exact", "act_to_first_dev_redo", "conformance_severity", "resource_NONE_perc", "resource_user_perc", "resource_batch_perc"]
target = ps_orig.ConformanceTarget("conformance_severity")
searchspace = ps_orig.create_selectors(data, ignore=ignore)
task = ps_orig.SubgroupDiscoveryTask (
    data,
    target,
    searchspace,
    result_set_size=10,
    depth=1,
    qf=ps_orig.AUCQFNumeric(a=0.5, comparison="complement", direction = "less_compliant", min_sg_coverage=0.02, max_sg_coverage=0.8))
result_exp_4 = ps_orig.BeamSearch(beam_width_adaptive = True).execute(task)
emm_result_exp_4 = result_exp_4.to_dataframe()
emm_result_exp_4_dedup = dedup_by_cover(result_exp_4, emm_result_exp_4)

In [18]:
emm_result_exp_4_dedup

,quality,subgroup,size_sg,size_dataset,size_complement,coverage_sg,auc_vs_dataset,auc_vs_complement,emd_vs_dataset,emd_vs_complement,...,median_sg,median_complement,median_dataset,dist_sg,dist_complement,dist_dataset,perfect_frac_sg,perfect_frac_dataset,worst_frac_sg,worst_frac_dataset
0,0.211080,case:Spend area text=='Packaging',76614,182998,106384,0.418660,0.594824,0.663112,0.636680,1.095194,...,4.0,3.0,3.0,"{0: 0.15840446915707312, 1: 0.0004698880100242...","{0: 0.3585783576477666, 1: 0.00759512708677996...","{0: 0.2747734947922928, 1: 0.00461207226308484...",0.158404,0.274773,0.387436,0.277315
1,0.207539,case:Spend classification text=='PR',115048,182998,67950,0.628684,0.548596,0.630874,0.345284,0.929895,...,4.0,3.0,3.0,"{0: 0.20414088032821084, 1: 0.0028596759613378...","{0: 0.39436350257542313, 1: 0.0075791022810890...","{0: 0.2747734947922928, 1: 0.00461207226308484...",0.204141,0.274773,0.327628,0.277315
2,0.134960,case:Sub spend area text=='Labels',41696,182998,141302,0.227849,0.609157,0.641368,0.704925,0.912938,...,4.0,3.0,3.0,"{0: 0.15759305448963928, 1: 0.0002638142747505...","{0: 0.3093516015343024, 1: 0.00589517487367482...","{0: 0.2747734947922928, 1: 0.00461207226308484...",0.157593,0.274773,0.377398,0.277315
3,0.101680,case:Vendor=='vendorID_0120',10506,182998,172492,0.057410,0.700000,0.712182,1.309605,1.389369,...,4.0,3.0,3.0,"{0: 0.04673519893394251, 1: 9.518370454978108e...","{0: 0.2886626626162373, 1: 0.00488718317371240...","{0: 0.2747734947922928, 1: 0.00461207226308484...",0.046735,0.274773,0.459928,0.277315
4,0.096794,case:Item=='00001',4818,182998,178180,0.026328,0.790416,0.798269,1.785540,1.833821,...,5.0,3.0,3.0,"{0: 0.0, 1: 0.0, 2: 0.0, 3: 0.1579493565794935...","{0: 0.28220338983050847, 1: 0.0047367830283982...","{0: 0.2747734947922928, 1: 0.00461207226308484...",0.000000,0.274773,0.842051,0.277315
5,0.094962,case:Vendor=='vendorID_0103',3855,182998,179143,0.021066,0.820245,0.827137,1.897288,1.938116,...,5.0,3.0,3.0,"{0: 0.011932555123216601, 1: 0.0, 2: 0.0, 3: 0...","{0: 0.2804296009333326, 1: 0.00471132000692184...","{0: 0.2747734947922928, 1: 0.00461207226308484...",0.011933,0.274773,0.914137,0.277315
6,0.088015,case:Vendor=='vendorID_0104',7152,182998,175846,0.039082,0.713905,0.722604,1.409604,1.466935,...,5.0,3.0,3.0,"{0: 0.019854586129753916, 1: 0.0, 2: 0.0, 3: 0...","{0: 0.28514154430581307, 1: 0.0047996542429170...","{0: 0.2747734947922928, 1: 0.00461207226308484...",0.019855,0.274773,0.681907,0.277315
7,0.086727,throughput>=114.08,36616,182998,146382,0.200090,0.577545,0.596942,0.571260,0.714155,...,4.0,3.0,3.0,"{0: 0.14906051999126066, 1: 0.0027583570024033...","{0: 0.30621934390840405, 1: 0.0050757606809580...","{0: 0.2747734947922928, 1: 0.00461207226308484...",0.149061,0.274773,0.358313,0.277315
8,0.076981,case:Vendor=='vendorID_0106',5404,182998,177594,0.029530,0.717371,0.723985,1.379566,1.421544,...,5.0,3.0,3.0,"{0: 0.04496669133974834, 1: 0.0001850481125092...","{0: 0.2817662758876989, 1: 0.00474678198587790...","{0: 0.2747734947922928, 1: 0.00461207226308484...",0.044967,0.274773,0.674685,0.277315
9,0.069316,throughput: [88.04:114.08[,36601,182998,146397,0.200008,0.561997,0.577496,0.413452,0.516819,...,4.0,3.0,3.0,"{0: 0.19816398459058496, 1: 0.0036337804978006...","{0: 0.2939267881172428, 1: 0.00485665689870694...","{0: 0.2747734947922928, 1: 0.00461207226308484...",0.198164,0.274773,0.367121,0.277315
